# Day 2 — Supervisory analytics and the API

**Programme:** Operational Data Analytics and Dashboard Engineering for SupTech Systems
**Client context:** Bank of Tanzania (prototype only)

Day 1 turned a file of submissions into a curated dataset. Day 2 turns that
curated dataset into supervisory indicators, then into exceptions, and then
into the read model that the dashboard will consume.

The chain is the whole point:

```text
a submitted record  ->  a trustworthy record  ->  an indicator  ->  an exception  ->  a reason
```

This notebook is the readable reference implementation. The reusable one
lives in `day-02-supervisory-analytics/src/`. The two are written separately
on purpose, and `tests/test_day2_notebook_parity.py` checks that they agree
on every indicator and every exception.

> Everything in this repository is synthetic training data. The thresholds
> below are illustrative prototype values, not official supervisory limits.

## 0. Setup

The setup cell only prepares the environment and the file paths. It does not
touch the data.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

#: Where the repository lives. Discovered automatically, or supplied by the
#: caller when the notebook runs outside the repository.
SUPTECH_ROOT = globals().get("SUPTECH_ROOT") or os.environ.get("SUPTECH_ROOT") or None

#: Optional raw file base, for example
#: "https://raw.githubusercontent.com/your-org/your-repo/main".
REPO_RAW_BASE = os.environ.get("SUPTECH_REPO_RAW_BASE", "").strip()

#: The five files Day 2 consumes. Three of them are published by Day 1.
DATA_FILES = {
    "valid": "day-01-data-engineering/datasets/processed/valid_records.csv",
    "review": "day-01-data-engineering/datasets/processed/review_records.csv",
    "audit": "day-01-data-engineering/reports/data_quality_audit.csv",
    "institutions": "day-01-data-engineering/datasets/reference/institutions_registry.csv",
    "calendar": "day-01-data-engineering/datasets/reference/reporting_calendar.csv",
}

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # noqa: F821
except NameError:
    def display(*args, **kwargs):
        '''A fallback for plain Python, where IPython's display is absent.'''
        for item in args:
            print(item)

print("Running in Colab:", IN_COLAB)


def discover_root():
    """Walk up from the working directory looking for the repository."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "day-01-data-engineering").is_dir():
            return candidate
    return None


if SUPTECH_ROOT is None:
    SUPTECH_ROOT = discover_root()

print("Repository root:", SUPTECH_ROOT)

In [ ]:
# Colab only. In Colab the repository is usually not available, so upload the
# five files once. Locally this cell does nothing.
if IN_COLAB:
    from google.colab import files

    print("Upload these five files:")
    for name in DATA_FILES.values():
        print("  -", name)
    files.upload()

In [ ]:
def resolve_data_path(key):
    """Find one data file locally, then from a raw file base."""
    relative = DATA_FILES[key]

    if SUPTECH_ROOT is not None:
        candidate = Path(SUPTECH_ROOT) / relative
        if candidate.exists():
            return candidate

    if REPO_RAW_BASE:
        import urllib.request

        url = REPO_RAW_BASE.rstrip("/") + "/" + relative
        local = Path.cwd() / Path(relative).name
        urllib.request.urlretrieve(url, local)
        print("Downloaded", relative)
        return local

    raise FileNotFoundError(
        f"Could not find {relative}.\n"
        "Fix it in one of these ways:\n"
        "  1. Run this notebook from a clone of the repository.\n"
        "  2. Set SUPTECH_ROOT to the repository root.\n"
        "  3. Set SUPTECH_REPO_RAW_BASE to the raw file base of the repository.\n"
        "  4. In Colab, run the upload cell above and then re-run this cell."
    )


if "DATA_PATHS" not in globals():
    DATA_PATHS = {key: resolve_data_path(key) for key in DATA_FILES}

for key, path in DATA_PATHS.items():
    print(f"{key}: {path}")

### The curated layer comes from Day 1

Day 2 never reads the submitted file. It reads what Day 1 published: the
records that survived validation and the findings that explain them.

The `processed/` and `reports/` folders are generated output, so a fresh
clone will not have them yet. The next cell runs the Day 1 pipeline to
produce them. It is the same pipeline, with the same rules, and it writes
the same artefacts.

In [ ]:
missing = [key for key, path in DATA_PATHS.items() if not Path(path).exists()]

if missing:
    if SUPTECH_ROOT is None:
        raise FileNotFoundError(
            "The Day 1 output is missing and the repository was not found.\n"
            "Run the upload cell and resolve the paths again, or run the Day 1 "
            "pipeline locally:\n"
            "    python day-01-data-engineering/src/validation.py"
        )
    print("Day 1 output is missing for:", missing)
    print("Running the Day 1 pipeline to publish it.")

    day1_src = Path(SUPTECH_ROOT) / "day-01-data-engineering" / "src"
    if str(day1_src) not in sys.path:
        sys.path.insert(0, str(day1_src))

    import validation

    result = validation.run_pipeline()
    validation.write_outputs(result)
    print("Day 1 artefacts published.")
else:
    print("Day 1 output is already in place.")

for key, path in DATA_PATHS.items():
    print(f"{key}: {Path(path).exists()}")

## 1. The curated layer

Seven records survived validation for the period 2026-11. The other
eighteen rows of the submitted file are not here: seventeen were rejected
and one was an exact duplicate. They live in the audit trail, not in the
curated layer, and no indicator is computed from them.

In [ ]:
def read_csv(path):
    """Read a published file as text, without repairing anything."""
    frame = pd.read_csv(path, dtype="string", keep_default_na=True)
    frame.columns = [str(column).strip().lower() for column in frame.columns]
    return frame


valid = read_csv(DATA_PATHS["valid"])
review = read_csv(DATA_PATHS["review"])
audit = read_csv(DATA_PATHS["audit"])
institutions = read_csv(DATA_PATHS["institutions"])
calendar = read_csv(DATA_PATHS["calendar"])

records = pd.concat([valid, review], ignore_index=True)
records["total_deposits"] = pd.to_numeric(records["total_deposits"], errors="coerce")
records["total_loans"] = pd.to_numeric(records["total_loans"], errors="coerce")

print("Curated records:", len(records))
print("Reporting periods:", sorted(set(records["reporting_period"].dropna())))
print()
display(records[[
    "source_row_number", "institution_id", "reporting_period",
    "total_deposits", "total_loans", "validation_status",
]].to_string(index=False))

### The findings that explain the curated layer

Only three of the seven records carry a finding, and those findings are
what a supervisor sees next to an exception later.

In [ ]:
print("Findings in the audit trail:", len(audit))
display(
    audit[["source_row_number", "institution_id", "rule_id", "rule_name",
           "severity", "observed_value"]].to_string(index=False)
)

## 2. The indicator catalog

An indicator is a measure. It says what the numbers are, and it carries no
judgement: a loan-to-deposit ratio of 1.4 is a number, and only a threshold
decides whether it is a problem.

The grain is declared. An indicator is computed for one institution in one
period, or for the whole sector in one period, and a consumer never has to
guess which one a number is.

In [ ]:
INDICATOR_CATALOG = pd.DataFrame(
    [
        ("IND-001", "total_deposits", "institution", "TZS", 2,
         "Total customer deposits reported for the period.",
         "total_deposits of the curated record"),
        ("IND-002", "total_loans", "institution", "TZS", 2,
         "Total loans and advances reported for the period.",
         "total_loans of the curated record"),
        ("IND-003", "loan_to_deposit_ratio", "institution", "ratio", 4,
         "Loans as a share of deposits. The core intermediation indicator.",
         "total_loans / total_deposits"),
        ("IND-004", "sector_deposit_share", "institution", "percent", 4,
         "The institution's share of all deposits reported for the period.",
         "total_deposits / sector_total_deposits * 100"),
        ("IND-005", "deposit_growth_period_on_period", "institution", "percent", 4,
         "Change in deposits against the institution's previous period.",
         "(deposits - previous deposits) / previous deposits * 100"),
        ("IND-006", "loan_growth_period_on_period", "institution", "percent", 4,
         "Change in loans against the institution's previous period.",
         "(loans - previous loans) / previous loans * 100"),
        ("IND-007", "sector_total_deposits", "sector", "TZS", 2,
         "Total deposits reported across the supervised sector.",
         "sum(total_deposits) over the curated records of the period"),
        ("IND-008", "sector_total_loans", "sector", "TZS", 2,
         "Total loans reported across the supervised sector.",
         "sum(total_loans) over the curated records of the period"),
        ("IND-009", "sector_loan_to_deposit_ratio", "sector", "ratio", 4,
         "Sector loans as a share of sector deposits.",
         "sector_total_loans / sector_total_deposits"),
        ("IND-010", "sector_deposit_concentration_hhi", "sector", "index", 4,
         "Herfindahl-Hirschman concentration index of sector deposits.",
         "sum((deposit share / 100) ** 2) * 10000"),
        ("IND-011", "sector_watchlist_exposure_count", "sector", "count", 0,
         "Curated records from an institution on the enhanced monitoring list.",
         "count of curated records whose institution is on the watchlist"),
        ("IND-012", "sector_reporting_coverage_ratio", "sector", "percent", 4,
         "Share of the institutions expected to report that reached the "
         "curated layer.",
         "reporting institutions / expected reporters * 100"),
    ],
    columns=["indicator_id", "indicator_name", "grain", "unit", "decimals",
             "description", "formula"],
)

print(len(INDICATOR_CATALOG), "declared indicators")
display(INDICATOR_CATALOG[["indicator_id", "indicator_name", "grain", "unit",
                            "decimals"]].to_string(index=False))

## 3. The threshold catalog

A threshold is a declared limit on one indicator, with a severity and the
reason a supervisor would give for caring.

Two things about the severities:

* They are **not** the Day 1 severities. Day 1 asked *can this record be
  trusted* and answered `BLOCK` or `WARN`. Day 2 asks *does this number
  deserve attention* and answers `CRITICAL`, `HIGH`, `MEDIUM` or `LOW`.
* The bands of one indicator do not overlap, so an indicator raises at most
  one exception for one subject.

The `explaining_rules` column names the Day 1 findings that explain a
breach. A threshold with no rule explains nothing through the audit trail,
because the risk it measures is not a symptom of a data quality problem.

In [ ]:
THRESHOLD_CATALOG = pd.DataFrame(
    [
        ("THR-001", "ldr_elevated", "IND-003", "institution", "HIGH",
         0.9, 1.5, False, True, "record", "W001",
         "Loan-to-deposit ratio is elevated"),
        ("THR-002", "ldr_critical", "IND-003", "institution", "CRITICAL",
         1.5, None, False, False, "record", "W001",
         "Loan-to-deposit ratio is critically high"),
        ("THR-003", "zero_deposits", "IND-001", "institution", "CRITICAL",
         None, 0.0, False, True, "record", "W002",
         "Institution reported zero deposits"),
        ("THR-004", "deposit_share_concentration", "IND-004", "institution",
         "HIGH", 25.0, None, False, False, "record", "",
         "Institution holds a large share of sector deposits"),
        ("THR-005", "deposit_growth_collapse", "IND-005", "institution", "HIGH",
         None, -20.0, False, False, "record", "",
         "Deposits fell sharply against the previous period"),
        ("THR-006", "loan_growth_spike", "IND-006", "institution", "MEDIUM",
         50.0, None, False, False, "record", "",
         "Loans grew by more than 50% against the previous period"),
        ("THR-007", "sector_loan_to_deposit_above", "IND-009", "sector", "HIGH",
         1.0, None, False, False, "period", "W001",
         "Sector loans exceed sector deposits"),
        ("THR-008", "sector_deposit_concentration_elevated", "IND-010",
         "sector", "MEDIUM", 2000.0, None, False, False, "period", "",
         "Sector deposit market is moderately concentrated"),
        ("THR-009", "sector_watchlist_exposure", "IND-011", "sector", "HIGH",
         0.0, None, False, False, "period", "W003",
         "Curated records include watchlist institutions"),
        ("THR-010", "sector_reporting_coverage_low", "IND-012", "sector",
         "MEDIUM", None, 80.0, False, False, "period", "",
         "Fewer than 80% of expected institutions reported"),
    ],
    columns=["threshold_id", "threshold_name", "indicator_id", "grain",
             "severity", "lower_bound", "upper_bound", "lower_inclusive",
             "upper_inclusive", "link_mode", "explaining_rules", "message"],
)

print(len(THRESHOLD_CATALOG), "declared thresholds")
display(THRESHOLD_CATALOG[["threshold_id", "threshold_name", "indicator_id",
                            "grain", "severity", "lower_bound", "upper_bound",
                            "link_mode"]].to_string(index=False))

## 4. Reference implementation

This is a readable reference implementation of the Day 2 analytics, written
to be read rather than to be fast. The production version lives in
`day-02-supervisory-analytics/src/indicators.py` and
`day-02-supervisory-analytics/src/exceptions.py`.

Two decisions are worth pointing at while you read it.

**An indicator that cannot be computed is null, not zero.** The
loan-to-deposit ratio of an institution that reports zero deposits is
undefined. It is not infinite and it is not zero. The same is true of a
growth rate for an institution with no earlier period.

**The previous period is the previous period that exists.** The growth rate
is computed against the last period in the curated layer for that
institution, not against an imagined one.

In [ ]:
QUARTER_END_MONTHS = (3, 6, 9, 12)


def expected_reporters(period, registry):
    """The institutions expected to file for one period, from the register."""
    month = int(str(period).split("-")[1])
    frequency = registry["reporting_frequency"].str.strip().str.lower()
    monthly = set(registry.loc[frequency == "monthly", "institution_id"])
    quarterly = set(registry.loc[frequency == "quarterly", "institution_id"])
    if month in QUARTER_END_MONTHS:
        return monthly | quarterly
    return monthly


def previous_records(records):
    """Map (institution, period) to the previous record of that institution."""
    earlier = {}
    for institution_id, group in records.groupby("institution_id"):
        ordered = group.sort_values("reporting_period")
        rows = list(ordered.iterrows())
        for index in range(1, len(rows)):
            key = (rows[index][1]["institution_id"], rows[index][1]["reporting_period"])
            earlier[key] = rows[index - 1][1]
    return earlier


def compute_indicators(records, registry, catalog):
    """Compute every declared indicator over the curated layer."""
    rows = []
    watchlist = set(
        registry.loc[
            registry["watchlist_flag"].str.strip().str.lower() == "true",
            "institution_id",
        ]
    )
    earlier_by_key = previous_records(records)

    def add(indicator_id, period, subject_id, value, institution_id=None):
        decimals = int(
            catalog.loc[catalog["indicator_id"] == indicator_id, "decimals"].iloc[0]
        )
        rows.append({
            "indicator_id": indicator_id,
            "reporting_period": period,
            "subject_id": subject_id,
            "institution_id": institution_id,
            "indicator_value": None if value is None else round(float(value), decimals),
        })

    for period in sorted(records["reporting_period"].dropna().unique()):
        period_records = records[records["reporting_period"] == period]
        sector_deposits = period_records["total_deposits"].sum()
        sector_loans = period_records["total_loans"].sum()

        for _, record in period_records.iterrows():
            institution_id = record["institution_id"]
            deposits = record["total_deposits"]
            loans = record["total_loans"]

            add("IND-001", period, institution_id, deposits, institution_id)
            add("IND-002", period, institution_id, loans, institution_id)

            ratio = None
            if pd.notna(deposits) and deposits != 0:
                ratio = loans / deposits
            add("IND-003", period, institution_id, ratio, institution_id)

            share = None
            if sector_deposits:
                share = deposits / sector_deposits * 100
            add("IND-004", period, institution_id, share, institution_id)

            earlier = earlier_by_key.get((institution_id, period))
            deposit_growth = None
            loan_growth = None
            if earlier is not None and earlier["total_deposits"]:
                deposit_growth = (
                    (deposits - earlier["total_deposits"])
                    / earlier["total_deposits"] * 100
                )
            if earlier is not None and earlier["total_loans"]:
                loan_growth = (
                    (loans - earlier["total_loans"]) / earlier["total_loans"] * 100
                )
            add("IND-005", period, institution_id, deposit_growth, institution_id)
            add("IND-006", period, institution_id, loan_growth, institution_id)

        add("IND-007", period, "SECTOR", sector_deposits)
        add("IND-008", period, "SECTOR", sector_loans)

        sector_ratio = None
        if sector_deposits:
            sector_ratio = sector_loans / sector_deposits
        add("IND-009", period, "SECTOR", sector_ratio)

        concentration = None
        if sector_deposits:
            shares = [value / sector_deposits for value in period_records["total_deposits"]]
            concentration = sum(share ** 2 for share in shares) * 10000
        add("IND-010", period, "SECTOR", concentration)

        exposure = int(
            period_records["institution_id"].isin(watchlist).sum()
        )
        add("IND-011", period, "SECTOR", exposure)

        reporting = set(period_records["institution_id"].dropna())
        expected = expected_reporters(period, registry)
        coverage = None
        if expected:
            coverage = len(reporting) / len(expected) * 100
        add("IND-012", period, "SECTOR", coverage)

    return pd.DataFrame(rows)


indicators = compute_indicators(records, institutions, INDICATOR_CATALOG)
print(len(indicators), "indicator values")
display(
    indicators[indicators["reporting_period"] == "2026-11"]
    .pivot_table(index="subject_id", columns="indicator_id",
                 values="indicator_value")
    .to_string()
)

### Thresholds into exceptions

Each threshold is a band. A value inside the band raises an exception, and
the exception carries the observed value, the limit and the severity.

The identifier of an exception is built from the threshold, the subject and
the period, so the same exception has the same identifier in every run.

In [ ]:
def evaluate_thresholds(indicators, catalog):
    """Evaluate every declared threshold against the computed indicators."""
    rows = []
    for _, threshold in catalog.iterrows():
        matching = indicators[
            indicators["indicator_id"] == threshold["indicator_id"]
        ]
        for _, indicator_row in matching.iterrows():
            value = indicator_row["indicator_value"]
            if value is None or pd.isna(value):
                continue

            breach = True
            lower = threshold["lower_bound"]
            upper = threshold["upper_bound"]
            if pd.notna(lower):
                breach = breach and (
                    value >= lower if threshold["lower_inclusive"] else value > lower
                )
            if pd.notna(upper):
                breach = breach and (
                    value <= upper if threshold["upper_inclusive"] else value < upper
                )
            if not breach:
                continue

            subject_id = indicator_row["subject_id"]
            period = indicator_row["reporting_period"]
            rows.append({
                "exception_id": f"EXC-{threshold['threshold_id']}-{subject_id}-{period}",
                "threshold_id": threshold["threshold_id"],
                "threshold_name": threshold["threshold_name"],
                "indicator_id": threshold["indicator_id"],
                "grain": threshold["grain"],
                "reporting_period": period,
                "subject_id": subject_id,
                "institution_id": indicator_row["institution_id"],
                "severity": threshold["severity"],
                "observed_value": float(value),
                "limit_description": describe_limit(threshold),
                "message": threshold["message"],
                "link_mode": threshold["link_mode"],
            })

    exceptions = pd.DataFrame(rows)
    if len(exceptions):
        order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
        exceptions = (
            exceptions.assign(_rank=exceptions["severity"].map(order))
            .sort_values(["_rank", "reporting_period", "indicator_id", "subject_id"])
            .drop(columns=["_rank"])
            .reset_index(drop=True)
        )
    return exceptions


def describe_limit(threshold):
    """Describe a threshold band in words."""
    lower = threshold["lower_bound"]
    upper = threshold["upper_bound"]

    def number(value):
        return str(int(value)) if float(value) == int(value) else str(value)

    if pd.isna(lower) and pd.isna(upper):
        return "any value"
    if pd.isna(lower):
        edge = "at or below" if threshold["upper_inclusive"] else "below"
        return f"{edge} {number(upper)}"
    if pd.isna(upper):
        edge = "at or above" if threshold["lower_inclusive"] else "above"
        return f"{edge} {number(lower)}"
    lower_edge = "at or above" if threshold["lower_inclusive"] else "above"
    upper_edge = "at or below" if threshold["upper_inclusive"] else "below"
    return f"{lower_edge} {number(lower)} and {upper_edge} {number(upper)}"


exceptions = evaluate_thresholds(indicators, THRESHOLD_CATALOG)
print(len(exceptions), "exceptions")
display(exceptions[["exception_id", "indicator_id", "subject_id", "severity",
                    "observed_value", "limit_description"]].to_string(index=False))

## 5. The exceptions and the audit trail

An exception is never a number on its own. It carries the threshold
evidence, the reason a supervisor would give, and the data quality findings
that explain it.

Look at the two institution exceptions:

* **TZ013** was flagged by Day 1 for a loan-to-deposit ratio out of band
  (`W001`), and the critical ratio exception is explained by that finding.
* **TZ002** holds more than a quarter of the sector's deposits and its
  record was perfectly clean. There is no finding to attach, and the
  exception says so rather than borrowing an unrelated one.

In [ ]:
print("Findings for the records behind the exceptions:")
display(
    audit[["source_row_number", "institution_id", "rule_id", "rule_name",
           "severity", "detail"]].to_string(index=False)
)

print()
print("Severity counts:")
display(
    exceptions["severity"].value_counts().reindex(
        ["CRITICAL", "HIGH", "MEDIUM", "LOW"], fill_value=0
    ).to_frame("exceptions")
)

## 6. Export

The outputs are written in the shape the API and the database loader
consume.

In [ ]:
OUTPUT_DIR = Path("day2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

indicators.to_csv(OUTPUT_DIR / "indicator_values.csv", index=False)
exceptions.to_csv(OUTPUT_DIR / "exceptions.csv", index=False)

print("Written:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("  -", path, f"({len(pd.read_csv(path))} rows)")

## 7. Self-check

Run this cell before committing any change. Every line has to pass.

The numbers below are the contract for the shipped dataset: seven curated
records, six exceptions, and the sector totals computed by hand.

In [ ]:
checks = []


def check(label, condition, detail=""):
    checks.append((label, bool(condition), detail))
    print(f"{'PASS' if condition else 'FAIL'}  {label}  {detail}")


curated = records[records["reporting_period"] == "2026-11"]

check("seven curated records", len(curated) == 7, f"got {len(curated)}")
check("six exceptions", len(exceptions) == 6, f"got {len(exceptions)}")
check(
    "every declared indicator was computed",
    set(indicators["indicator_id"]) == set(INDICATOR_CATALOG["indicator_id"]),
)
check(
    "the sector holds 2,110,000,000 in deposits",
    abs(indicators.query("indicator_id == 'IND-007'")["indicator_value"].iloc[0]
        - 2_110_000_000) < 1,
)
check(
    "the sector holds 1,580,000,000 in loans",
    abs(indicators.query("indicator_id == 'IND-008'")["indicator_value"].iloc[0]
        - 1_580_000_000) < 1,
)
check(
    "TZ013 has a ratio of 9",
    abs(indicators.query(
        "indicator_id == 'IND-003' and subject_id == 'TZ013'"
    )["indicator_value"].iloc[0] - 9) < 0.0001,
)
check(
    "the ratio of an institution with zero deposits is null",
    indicators.query(
        "indicator_id == 'IND-003' and subject_id == 'TZ014'"
    )["indicator_value"].isna().all(),
)
check(
    "TZ013's exception is critical",
    "EXC-THR-002-TZ013-2026-11" in set(exceptions["exception_id"]),
)

print()
failed = [label for label, passed, _ in checks if not passed]
print(f"{len(checks) - len(failed)} of {len(checks)} checks passed.")
if failed:
    print("Failed:", failed)

## Extension exercises

Work in the cells below. Do not delete the reference implementation: it stays
visible so you can compare your version with it.

1. **Add an indicator.** Declare it in `INDICATOR_CATALOG`, compute it in
   `compute_indicators`, then add a threshold for it in
   `THRESHOLD_CATALOG`. The exception appears on its own.
2. **Add a threshold.** Declare it in `THRESHOLD_CATALOG` and nothing else.
   A threshold is data, not code.
3. **Change the severity of a threshold** and watch the exception list
   reorder. Severity is a declaration, not a decision inside the code.
4. **Break a band on purpose.** Make two bands overlap and see what
   happens. Then write down what the bands are supposed to mean.

## Completion criteria

- [ ] The curated layer was read from the Day 1 artefacts.
- [ ] Every declared indicator was computed.
- [ ] An indicator that cannot be computed came out null.
- [ ] The thresholds raised the expected exceptions.
- [ ] Every exception carries its evidence.
- [ ] The self-check passes.